In [2]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

data_ev=pd.read_json("Script_generador_tickets_sinteticos_electricos/data/tickets_ev_sinteticos.json")

In [4]:
data_gas=pd.read_json("Script_generador_tickets_sinteticos_gasolina/data/tickets/tickets_sinteticos.json",convert_dates=True )

In [5]:
data_gas

,idTicket,idEmpresa,empresaNombre,idUsuario,fechaEmision,horaEmision,metodoPago,estacion,lineas,baseImponible,iva,total,moneda,tipoDocumento
0,T-86837D3A0D41,EMP001,Transporte_01 S.L.,EMP001-U1,2025-04-13,04:23:15,Tarjeta crédito,"{'id': '1492', 'nombre': 'COOP. LA SIBERIA EXT...","[{'producto': 'Gasóleo A', 'litros': 17.44, 'p...",21.04,4.42,25.46,EUR,Factura simplificada
1,T-9D71DD5BDFA3,EMP001,Transporte_01 S.L.,EMP001-U1,2025-06-14,00:57:19,Tarjeta crédito,"{'id': '6170', 'nombre': 'E.S SANT ISIDRE', 'p...","[{'producto': 'Gasóleo A', 'litros': 25.64, 'p...",29.69,6.23,35.92,EUR,Factura simplificada
2,T-A7F04C34CBC5,EMP001,Transporte_01 S.L.,EMP001-U1,2025-01-03,13:38:09,Efectivo,"{'id': '3956', 'nombre': 'REPSOL', 'provincia'...","[{'producto': 'Gasóleo A', 'litros': 50.44, 'p...",60.28,12.66,72.94,EUR,Factura simplificada
3,T-8ACF37775D56,EMP001,Transporte_01 S.L.,EMP001-U1,2025-06-04,00:23:23,Tarjeta crédito,"{'id': '550', 'nombre': 'GLOBAL OIL', 'provinc...","[{'producto': 'Gasóleo A', 'litros': 23.02, 'p...",27.05,5.68,32.73,EUR,Factura simplificada
4,T-F06D43C88D8A,EMP001,Transporte_01 S.L.,EMP001-U1,2025-07-26,17:43:18,Tarjeta crédito,"{'id': '451', 'nombre': 'BP', 'provincia': 'AL...","[{'producto': 'Gasóleo A', 'litros': 52.21, 'p...",62.01,13.02,75.03,EUR,Factura simplificada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,T-A7E2C901EED5,EMP003,Transporte_03 S.L.,EMP003-U27,2025-05-19,06:20:08,Efectivo,"{'id': '148', 'nombre': 'SAN ISIDRO', 'provinc...","[{'producto': 'Gasolina 95 E5', 'litros': 69.8...",88.31,18.54,106.85,EUR,Factura simplificada
1346,T-4DBB2F0DABB2,EMP003,Transporte_03 S.L.,EMP003-U27,2025-02-24,13:03:03,Tarjeta crédito,"{'id': '9203', 'nombre': 'REPSOL', 'provincia'...","[{'producto': 'Gasolina 95 E5', 'litros': 64.8...",81.20,17.05,98.25,EUR,Factura simplificada
1347,T-408DC754E9F6,EMP003,Transporte_03 S.L.,EMP003-U27,2025-05-04,08:47:00,Tarjeta crédito,"{'id': '6427', 'nombre': 'E.S. POLIGONO', 'pro...","[{'producto': 'Gasolina 95 E5', 'litros': 46.5...",58.64,12.32,70.96,EUR,Factura simplificada
1348,T-CC6CD193ECE5,EMP003,Transporte_03 S.L.,EMP003-U27,2025-02-28,05:26:07,Tarjeta crédito,"{'id': '6154', 'nombre': 'REPSOL', 'provincia'...","[{'producto': 'Gasolina 95 E5', 'litros': 47.1...",58.17,12.21,70.38,EUR,Factura simplificada


In [6]:
data_gas["lineas"]

0       [{'producto': 'Gasóleo A', 'litros': 17.44, 'p...
1       [{'producto': 'Gasóleo A', 'litros': 25.64, 'p...
2       [{'producto': 'Gasóleo A', 'litros': 50.44, 'p...
3       [{'producto': 'Gasóleo A', 'litros': 23.02, 'p...
4       [{'producto': 'Gasóleo A', 'litros': 52.21, 'p...
                              ...                        
1345    [{'producto': 'Gasolina 95 E5', 'litros': 69.8...
1346    [{'producto': 'Gasolina 95 E5', 'litros': 64.8...
1347    [{'producto': 'Gasolina 95 E5', 'litros': 46.5...
1348    [{'producto': 'Gasolina 95 E5', 'litros': 47.1...
1349    [{'producto': 'Gasolina 95 E5', 'litros': 32.0...
Name: lineas, Length: 1350, dtype: object

In [7]:
estacion_df = pd.json_normalize(data_gas["estacion"])
data_gas = data_gas.drop(columns=["estacion"]).join(estacion_df.add_prefix("estacion."))


In [8]:
df_exploded = data_gas.explode("lineas").reset_index(drop=True)

# Normaliza la columna "lineas" (dict → columnas)
lineas_normalizadas = pd.json_normalize(df_exploded["lineas"])

# Une las nuevas columnas con el dataset original
data_gas = df_exploded.drop(columns=["lineas"]).join(lineas_normalizadas.add_prefix("lineas."))

In [9]:
data_gas["fechaEmision"]=pd.to_datetime(data_gas["fechaEmision"])
data_gas["horaEmision"]=pd.to_datetime(data_gas["horaEmision"], format="%H:%M:%S")

In [10]:
data_gas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1350 entries, 0 to 1349
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   idTicket               1350 non-null   object        
 1   idEmpresa              1350 non-null   object        
 2   empresaNombre          1350 non-null   object        
 3   idUsuario              1350 non-null   object        
 4   fechaEmision           1350 non-null   datetime64[ns]
 5   horaEmision            1350 non-null   datetime64[ns]
 6   metodoPago             1350 non-null   object        
 7   baseImponible          1350 non-null   float64       
 8   iva                    1350 non-null   float64       
 9   total                  1350 non-null   float64       
 10  moneda                 1350 non-null   object        
 11  tipoDocumento          1350 non-null   object        
 12  estacion.id            1350 non-null   object        
 13  est

In [11]:
data_gas['empresaNombre']


0       Transporte_01 S.L.
1       Transporte_01 S.L.
2       Transporte_01 S.L.
3       Transporte_01 S.L.
4       Transporte_01 S.L.
               ...        
1345    Transporte_03 S.L.
1346    Transporte_03 S.L.
1347    Transporte_03 S.L.
1348    Transporte_03 S.L.
1349    Transporte_03 S.L.
Name: empresaNombre, Length: 1350, dtype: object

In [12]:
[['empresaNombre', 'idUsuario', 'fechaEmision', 'horaEmision', 'metodoPago', 'baseImponible', 'total', 'estacion.id', 'estacion.provincia']]

[['empresaNombre',
  'idUsuario',
  'fechaEmision',
  'horaEmision',
  'metodoPago',
  'baseImponible',
  'total',
  'estacion.id',
  'estacion.provincia']]

In [13]:
from skrub import TableReport

TableReport(data_gas) 

Processing column  25 / 25


,,,,,,,,,,,,,,,,,,,,,,,,,


In [14]:
data_gas["idEmpresa"].value_counts()

idEmpresa
EMP001    450
EMP002    450
EMP003    450
Name: count, dtype: int64

In [15]:
data_gas['estacion.nombre'].value_counts()

estacion.nombre
REPSOL                     317
CEPSA                       75
MOEVE                       57
GALP                        57
BALLENOIL                   45
                          ... 
MOEVE INGENIO                1
BENZINERA GRANOLLERS         1
MERCASOSA                    1
CARBURANTES IBIZA, S.L.      1
COPVILAR                     1
Name: count, Length: 513, dtype: int64

In [16]:
data_gas.fechaEmision.max()

Timestamp('2025-07-31 00:00:00')

Abril es el maes eque mas se reponsto
El metodo de pago mas comuns es la tarjete de empresa
Las fechas van desde enero a 31 julio
Lo normal es gastar 63 euros en gasoluna ( o unos 42 litros )
Repsos es la esacion mas comuns

In [17]:
data_gas['lineas.producto'].value_counts()

lineas.producto
Gasolina 95 E5     550
Gasóleo A          400
Gasolina 98 E5     200
Gasóleo Premium    200
Name: count, dtype: int64

In [18]:
TableReport(data_gas[data_gas['estacion.nombre']=="REPSOL"])

Processing column  25 / 25


,,,,,,,,,,,,,,,,,,,,,,,,,


Repsol no es la mas barata al contrario de lo esperaple

In [19]:
data_gas.groupby('estacion.nombre')["lineas.precioUnitario"].median()

estacion.nombre
 E.S. LA TORRETA               1.410
 REPSOL UBEDA                  1.515
(SIN RÓTULO)                   1.527
0                              1.432
A.N. ENERGETICOS-ALDEANUEVA    1.523
                               ...  
ZARCAR                         1.538
ZD URBAN VIC                   1.482
ZONA AUTO LUAIDE AVIA          1.495
ZONA DIESEL                    1.517
ÁREA 117                       1.487
Name: lineas.precioUnitario, Length: 513, dtype: float64

Lo mas barato es Villager

In [20]:
data_gas.groupby('lineas.producto')["lineas.precioUnitario"].median()

lineas.producto
Gasolina 95 E5     1.5190
Gasolina 98 E5     1.6990
Gasóleo A          1.4370
Gasóleo Premium    1.5385
Name: lineas.precioUnitario, dtype: float64

In [21]:
data_gas.groupby('estacion.grupo')["lineas.precioUnitario"].median()

estacion.grupo
 E.S. LA TORRETA               1.410
 REPSOL UBEDA                  1.515
(SIN RÓTULO)                   1.527
0                              1.432
A.N. ENERGETICOS-ALDEANUEVA    1.523
                               ...  
ZARCAR                         1.538
ZD URBAN VIC                   1.482
ZONA AUTO LUAIDE AVIA          1.495
ZONA DIESEL                    1.517
ÁREA 117                       1.487
Name: lineas.precioUnitario, Length: 513, dtype: float64

Lo mas barato es el gasoleo A

In [22]:
import folium
from folium import plugins
mapa = folium.Map(location=[40, -5], zoom_start=6, )
for x in range(len(data_gas)):
         
    
    folium.Marker(location=data_gas.iloc[x][["estacion.lat","estacion.lon"]], popup=data_gas.iloc[x]["lineas.precioUnitario"]
              
              ).add_to(mapa)

mapa


c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])
c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])
c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behav

In [23]:
import folium
from folium import plugins
mapa = folium.Map(location=[40, -5], zoom_start=6, )
for x in range(len(data_gas)):
         
    
    folium.Marker(location=data_gas.iloc[x][["estacion.lat","estacion.lon"]], popup=data_gas.iloc[x]["lineas.precioUnitario"]
              
              ).add_to(mapa)

mapa

c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])
c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])
c:\Users\Jon\AppData\Local\Programs\Python\Python312\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behav

In [24]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# --- Helpers para elegir columnas disponibles ---
def pick_col(df, candidates):
    """Devuelve el primer nombre de columna que exista en df de la lista candidates."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_get(row, col, default="—"):
    return row[col] if (col and col in row.index and pd.notna(row[col])) else default

# Clave de estación
station_key = 'estacion.id' if 'estacion.id' in data_gas.columns else 'estacion.nombre'

# Precio medio por estación
precio_medio_por_est = (
    data_gas.groupby(station_key)['lineas.precioUnitario']
            .mean()
            .round(3)
            .to_dict()
)

# Detectar nombres reales de columnas para dirección y localidad
calle_col = pick_col(data_gas, [
    'estacion.calle', 'estacion.direccion', 'estacion.dirección',
    'estacion.dir', 'estacion.via', 'estacion.direccionCompleta'
])
loc_col = pick_col(data_gas, [
    'estacion.localidad', 'estacion.municipio', 'estacion.poblacion',
    'estacion.población', 'estacion.ciudad', 'estacion.localidadNombre'
])

# Otros metadatos frecuentes
nombre_col = pick_col(data_gas, ['estacion.nombre'])
grupo_col  = pick_col(data_gas, ['estacion.grupo', 'estacion.operador', 'estacion.empresa'])
lat_col    = pick_col(data_gas, ['estacion.lat', 'lat', 'latitude'])
lon_col    = pick_col(data_gas, ['estacion.lon', 'lon', 'longitude'])

print("Columnas detectadas →",
      f"nombre={nombre_col}, operador={grupo_col}, calle={calle_col}, localidad={loc_col}, lat={lat_col}, lon={lon_col}")

# Construir agregación con lo que exista
agg_dict = {}
for col in [nombre_col, grupo_col, calle_col, loc_col, lat_col, lon_col]:
    if col: agg_dict[col] = 'first'

# Si faltan lat/lon, no se puede mapear
if not lat_col or not lon_col:
    raise ValueError("No se encontraron columnas de lat/lon. Revisa que existan, p. ej. 'estacion.lat' y 'estacion.lon'.")

estaciones = data_gas.groupby(station_key).agg(agg_dict).reset_index()

# Añadir precio medio
estaciones['precio_medio'] = estaciones[station_key].map(precio_medio_por_est)

# Mapa + popups
mapa = folium.Map(location=[40, -5], zoom_start=6)
cluster = MarkerCluster().add_to(mapa)

for _, row in estaciones.iterrows():
    lat = safe_get(row, lat_col, None)
    lon = safe_get(row, lon_col, None)
    if lat is None or lon is None:
        continue

    nombre   = safe_get(row, nombre_col)
    operador = safe_get(row, grupo_col)
    calle    = safe_get(row, calle_col)
    loc      = safe_get(row, loc_col)
    pmedio   = row['precio_medio'] if pd.notna(row['precio_medio']) else None

    # Si no hay “calle” ni “localidad”, intenta componer algo con provincia si existe
    if calle == "—" and 'estacion.provincia' in estaciones.columns:
        calle = safe_get(row, 'estacion.provincia')

    popup_html = f"""
    <div style="font-size:14px; line-height:1.3">
        <b>{nombre}</b><br>
        Operador: {operador}<br>
        Dirección: {calle}<br>
        Localidad: {loc}<br>
        Precio medio: {f"{pmedio:.3f} €/l" if pmedio is not None else "—"}
    </div>
    """
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=nombre if nombre != "—" else "Estación"
    ).add_to(cluster)

mapa



Columnas detectadas → nombre=estacion.nombre, operador=estacion.grupo, calle=estacion.direccion, localidad=estacion.municipio, lat=estacion.lat, lon=estacion.lon
